In [ ]:
import pandas as pd
import numpy as np

train_df = pd.read_csv("../data/challenge1_train.csv")
test_df = pd.read_csv("../data/challenge1_test.csv")

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

train_df.head()

In [ ]:
print("Data types:")
print(train_df.dtypes)

print("\nMissing values:")
print(train_df.isnull().sum())

print("\nDuplicate rows:", train_df.duplicated().sum())

In [ ]:
X = train_df.drop(columns=["price", "ID"])
y = train_df["price"]

print("Features:")
print(X.columns.tolist())

print("\nTarget: price")
print("X shape:", X.shape)
print("y shape:", y.shape)

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("y_train:", y_train.shape)
print("y_val:", y_val.shape)

### Validation Strategy

We split the training dataset into 80% training data (5,918 observations) and 20% validation data (1,480 observations) using `random_state=42` for reproducibility.

We use RMSE as our main evaluation metric. The same validation split will be used to compare all four regression models fairly. Preprocessing is fitted only on the training data to prevent data leakage.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

numeric_features = [
    "bathrooms",
    "bedrooms",
    "square_feet",
    "latitude",
    "longitude"
]

categorical_features = [
    "category",
    "has_photo",
    "state"
]

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ]
)

print("Numerical features:", numeric_features)
print("Categorical features:", categorical_features)

## Linear Regression Model

Linear Regression is used as a baseline model to predict monthly apartment rental prices. We use the same preprocessing and validation split to evaluate the model using RMSE.

In [ ]:

from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
import numpy as np

# Linear Regression pipeline
linear_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LinearRegression())
])

# Train model
linear_pipeline.fit(X_train, y_train)

# Predictions
train_pred = linear_pipeline.predict(X_train)
val_pred = linear_pipeline.predict(X_val)

# RMSE
train_rmse = np.sqrt(mean_squared_error(y_train, train_pred))
val_rmse = np.sqrt(mean_squared_error(y_val, val_pred))

print("Linear Regression Results")
print(f"Training RMSE: {train_rmse:.2f}")
print(f"Validation RMSE: {val_rmse:.2f}")


In [ ]:

from pathlib import Path

# Save Linear Regression results
results = pd.DataFrame({
    "Model": ["Linear Regression"],
    "Train RMSE": [train_rmse],
    "Validation RMSE": [val_rmse]
})

Path("../results").mkdir(exist_ok=True)

results.to_csv(
    "../results/linear_regression_results.csv",
    index=False
)

display(results)


### Linear Regression Results

The Linear Regression model achieved a training RMSE of 515.01 and a validation RMSE of 549.97.

This model serves as our baseline. We will compare its performance with Decision Tree, Random Forest, and Gradient Boosting to determine which model performs best on the validation set.